In [ ]:
# from google.colab import drive
# drive.mount('/content/drive')
# %cd "/content/drive/MyDrive/"

import os
os.chdir(r"C:\Z")  # Cambia el directorio de trabajo
print(os.getcwd())  # Verifica que cambió correctamente el directorio base

In [ ]:
from SciServer import CasJobs as cj
import pandas as pd
import numpy as np
from concurrent.futures import ThreadPoolExecutor, as_completed
pd.set_option('display.max_rows', 20)

# Iniciar sesión
from SciServer import Authentication
Authentication.login(UserName = "Alberto_2002", Password = "#Blackie2002")

In [ ]:
# import os
# import requests
# from SciServer import CasJobs

# # Seleccionar la URL a la que está asociada el archivo fits
# sql_query = """
# SELECT TOP 15000
#     p.objid,
#     p.ra,
#     p.dec,
#     s.specObjID,
#     dbo.fGetUrlFitsSpectrum(s.specObjID) AS fits_url,
#     s.z AS redshift
# FROM 
#     PhotoObj AS p
# JOIN 
#     SpecObj AS s ON s.bestobjid = p.objid
# WHERE 
#     s.z BETWEEN 0.3 AND 0.7
#     AND s.zWarning = 0
# """

# sql_query = """
# SELECT TOP 1000000
#     p.objid,
#     p.ra,
#     p.dec,
#     s.specObjID,
#     dbo.fGetUrlFitsSpectrum(s.specObjID) AS fits_url,
#     s.z AS redshift
# FROM 
#     PhotoObj AS p
# JOIN 
#     SpecObj AS s ON s.bestobjid = p.objid
# WHERE 
#     s.z BETWEEN 0.3 AND 0.7
#     AND s.zWarning = 0
# """

# results = CasJobs.executeQuery(sql_query, context="DR18")
# fits_urls = results["fits_url"].tolist()

In [ ]:
import os
import requests
import time
import pandas as pd
from SciServer import CasJobs

# Parámetros iniciales
batch_size_inicial = 1000       # tamaño inicial del lote
min_batch_size = 1000           # tamaño mínimo de lote permitido
max_retries = 10                # máximo número de reintentos por lote
offset = 100000                 # registro inicial a recuperar
all_results = []
batch_results = []

while offset <= 1000000:
  
    current_batch_size = batch_size_inicial
    retries = 0
    success = False
    iteration_start = time.time()  # marca de tiempo para controlar el rate limit

    # Intentar ejecutar el query con el tamaño de lote actual. Si hay error, se reduce el lote y se reintenta.
    while not success and retries < max_retries:
        sql_query = f"""
        SELECT
            p.objid,
            s.specObjID,
            dbo.fGetUrlFitsSpectrum(s.specObjID) AS fits_url,
            s.z AS redshift
        FROM 
            PhotoObj AS p
        JOIN 
            SpecObj AS s ON s.bestobjid = p.objid
        WHERE 
            s.zWarning = 0
        ORDER BY p.objid
        OFFSET {offset} ROWS
        FETCH NEXT {current_batch_size} ROWS ONLY
        """
        try:
            batch_results = pd.DataFrame(CasJobs.executeQuery(sql_query, context="DR18"))
            success = True
            all_results.append(batch_results)
            offset += len(batch_results)
            print(f"Recuperados {offset} registros hasta ahora.")

        except Exception as e:
            print(f"Error en la consulta con batch_size = {current_batch_size}: {e}")
            # Reducir el tamaño del lote, pero sin bajar de min_batch_size
            current_batch_size = max(min_batch_size, current_batch_size // 2)
            retries += 1
            time.sleep(2)  # Esperar un poco antes de reintentar

    if retries == max_retries:
        print("No se pudo recuperar el batch tras varios reintentos. Terminando.")
        break

Error en la consulta con batch_size = 1000: Error when executing query. Http Response from CasJobs API returned status code 500:
{"Error Code":500,"Error Type":"InternalServerError","Error Message":"Failed to execute a query: Query exceeds queue time.  Please revise your query or use a longer queue.","LogMessageID":"1da927e0-70a4-4ef5-bdb8-7f54c93830b4"}


In [ ]:
# Combinar todos los resultados (suponiendo que cada batch es un DataFrame)
combined_results = pd.concat(all_results, ignore_index=True)
print(f"Total de registros recuperados: {len(combined_results)}")

# Extraer las URLs de los archivos FITS
fits_urls = combined_results["fits_url"].tolist()
print(f"Se han obtenido {len(fits_urls)} URLs de FITS.")

In [ ]:
import os
import requests
import time
import pandas as pd
from SciServer import CasJobs

# Directorio de descarga
output_dir = r"C:/Z/TFGF NO DRIVE/Python/spectrums"
os.makedirs(output_dir, exist_ok=True)

archivos_descargados = []

def download_file(url):
    filename = os.path.basename(url)
    local_path = os.path.join(output_dir, filename)
    
    # Saltar si ya se descargó este archivo
    if os.path.exists(local_path):
        print(f"Saltando {filename} (ya descargado).")
        return filename, None
    
    try:
        response = requests.get(url)
        response.raise_for_status()
        with open(local_path, "wb") as f:
            f.write(response.content)
        print(f"Descargado: {filename}")
        return filename, None
    except Exception as e:
        print(f"Error al descargar {url}: {e}")
        return filename, e

# Número de descargas en paralelo
paralelo = 100

with ThreadPoolExecutor(max_workers=paralelo) as executor:
    future_to_url = {executor.submit(download_file, url): url for url in fits_urls}
    
    for future in as_completed(future_to_url):
        filename, error = future.result()
        if error is None:
            archivos_descargados.append(filename)

print("Descarga finalizada")